# Aesthetic Score Optimization Example

This notebook demonstrates an approach to optimize for aesthetics by using a CLIP-based aesthetic scorer as an objective, as discussed in Section 5.1 of the [thesis](https://scholarsarchive.byu.edu/cgi/viewcontent.cgi?article=12256&context=etd#page=39.08).

In [ ]:
import os
import sys

if ".." not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

from examples.example_scenes import (
    BlenderManScene,
    CandleScene,
    CarScene,
    CarStudioScene,
    DinoScene,
    EinarScene,
    EinarSmallDomeScene,
    FlowerPotScene,
    HouseScene,
    RedCarScene,
    SciFiRobotScene,
    SpringPortraitScene,
    SpringPortraitSmallDomeScene,
    SpringScene,
)
from losses.aesthetic import (
    AestheticLossMaximize,
    AestheticLossWithTarget,
    LAIONAestheticScorer,
)
from utils.color.linear_to_srgb_converters import (
    LinearRec709ToAgXBase,
    SimpleGammaCurve,
)
from utils.color.tonemapping.agx_looks import AgXPunchyLook
from utils.optimize import optimize_with_criterion


In [ ]:
# Select the scene to optimize (uncomment the desired scene)
scene = SciFiRobotScene(device=device)
# scene = SpringScene(device=device)
# scene = CarScene(device=device)
# scene = BlenderManScene(device=device)
# scene = RedCarScene(device=device)
# scene = CandleScene(device=device)
# scene = HouseScene(device=device)
# scene = DinoScene(device=device)
# scene = FlowerPotScene(device=device)
# scene = CarStudioScene(configuration='dome_lights', device=device)
# scene = EinarScene(device=device)
# scene = EinarSmallDomeScene(device=device)
# scene = SpringPortraitScene(device=device)
# scene = SpringPortraitSmallDomeScene(device=device)


In [ ]:
# Hyperparameters
lr = 0.04
n_iter = 250
global_seed = 2

# The choice of color space converter can dramatically impact optimization results, as shown in Figure 5.1
color_space_converter = LinearRec709ToAgXBase(AgXPunchyLook())
# color_space_converter = SimpleGammaCurve()

# Initialize scorer and loss criterion
scorer = LAIONAestheticScorer(device=device)
criterion = AestheticLossMaximize(scorer)

# Or, in an attempt to reduce finding an adversarial solution, minimizing the difference of a target score can sometimes improve results:
# criterion = AestheticLossWithTarget(scorer, target_score=0.8)

optimize_with_criterion(
    scene,
    lr,
    n_iter,
    criterion,
    starting_multiplier_std=(0.1, 0.1, 0.1),
    output_subdirectory_name="aesthetic_score_example",
    n_results=4,
    render_color_space_converter=color_space_converter,
    require_physically_plausible_multipliers=True,
    title_prefix="Aesthetic Score Optimization",
    device=device,
    save_every=25,
    model_name="LAION sa_0_4_vit_b_32_linear",
    pretrained_source="LAION sa_0_4_vit_b_32_linear",
    seed=global_seed,
)
